# Playground: define, train and explore an experiment

A scratch notebook for the experiment pipeline (`fct.exp`), end to end: define an experiment from a
preset, train its settings here in the kernel with the log streaming below the cell, load one
trained setting and try analysis functions on it.

Edit the parameters cell and run top to bottom. Training resumes where it stopped and skips
finished settings, so re-running is cheap. `ROOT` holds the experiment; a different experiment in
the same `ROOT` is refused, so change `ROOT` (or delete it) to start another.

In [ ]:
# ROOT: experiment directory; None means $FCT_PLAYGROUND_ROOT, else <repo>/tmp/playground (git-ignored)
ROOT = None
# PRESET: the base preset, see fct.exp.preset_names()
PRESET = "tiny"
# OVERRIDES: flat ExpConfig fields (N, n_epochs, ...); put "device" here to pin the training device
OVERRIDES = {"n_epochs": 20000, "N": 32, "P": 256, "log_int": 2000, "save_int": 5000, "monitor_int": 2000}
# AXES: axis name -> values, one setting per combination; None keeps the preset's axes
AXES = {"net.L": [1, 2]}
# LOAD_DEVICE: where the net is loaded for analysis ("cpu", "cuda" or "auto"); not part of the experiment
LOAD_DEVICE = "cpu"
# SETTING: the setting to load below, an index or a dict of axis values such as {"net.L": 2}
SETTING = 0
# MEMBER: which network of an ensemble (net.B > 1) to load; 0 is also the only network when B = 1
MEMBER = 0

## Define the experiment

`OVERRIDES` are flat fields of the experiment (`N`, `n_epochs`), whereas `AXES` keys are dotted
paths into a setting's config (`net.L`, `train.lr`). The experiment is saved to `ROOT` before
training because `run_setting` and `select_setting` read it back from there.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

import fct
from fct.analyse import example_outputs, r_squared
from fct.config import resolve_device, resolve_dtype
from fct.data import get_dataset
from fct.exp import ExpConfig, expand, init_root, preset, run_setting, with_axes
from fct.spatial_exp import select_setting, settings_table
from fct.train import load_trained, select_member

REPO = Path(fct.__file__).resolve().parents[1]
ROOT = Path(ROOT or os.environ.get("FCT_PLAYGROUND_ROOT") or REPO / "tmp" / "playground")

In [ ]:
exp = ExpConfig.model_validate({**preset(PRESET).model_dump(), **OVERRIDES})
if AXES is not None:
    exp = with_axes(exp, AXES)
# The preset's description is about other sizes; describe this experiment instead.
exp = exp.model_copy(
    update={"description": f"Playground: preset {PRESET!r}, overrides {OVERRIDES}, axes {AXES}"}
)

# Raises if ROOT holds another experiment (or a legacy or orphaned one); never overwrites it.
init_root(exp, ROOT)
display(Markdown(settings_table(ROOT)))

## Train

Each setting trains in this kernel; the log lines appear as it runs.

In [ ]:
# run_setting resumes a partial run and skips a finished one, so re-running this cell is cheap.
settings = expand(exp)
for index in range(len(settings)):
    run_setting(exp, index, ROOT, log=print)

In [ ]:
def load_net_and_history(chosen, member=MEMBER):
    """The trained net and history of a done setting (one member of an ensemble)."""
    cfg = chosen.cfg
    device, dtype = resolve_device(cfg), resolve_dtype(cfg.dtype)
    net, history = load_trained(chosen.directory, cfg, device, dtype)
    return select_member(net, history, member)


fig, ax = plt.subplots()
for setting in settings:
    chosen = select_setting(ROOT, setting.axes, LOAD_DEVICE)
    if chosen.status != "done":
        print(f"skipping {dict(setting.axes)}: {chosen.status}")
        continue
    _, history = load_net_and_history(chosen)
    ax.plot(history["epoch"], history["mse"], label=str(dict(setting.axes)))
ax.set(xlabel="epoch", ylabel="mse", yscale="log")  # the total loss is dominated by the penalties
ax.legend()
plt.show()

## Load one trained setting

`SETTING` picks it, an index or axis values. `MEMBER` picks the network of an ensemble
(`net.B > 1`); a single network is member 0. The loader is written out here instead of using
`fct.spatial_exp.load_exp_run`, which also builds a test set that only exists for the linear
teacher.

In [ ]:
axes = settings[SETTING].axes if isinstance(SETTING, int) else SETTING
chosen = select_setting(ROOT, axes, LOAD_DEVICE)
if chosen.status != "done":
    raise RuntimeError(f"setting {dict(axes)} is {chosen.status}; run the train cell first")
cfg = chosen.cfg
net, history = load_net_and_history(chosen)
data = get_dataset(chosen.directory, cfg)
print(dict(axes), "| x", tuple(data.x.shape), "| y", tuple(data.y.shape))
print("history keys:", sorted(history))

## Analysis playground

`cfg`, `net`, `data` and `history` are loaded above. Entry points, with what each needs (see `help(...)`):

- `fct.analyse`: any trained net and dataset. `example_outputs`, `pick_examples`, `r_squared`, `r_squared_per_output`, `prediction_report`, `grid_predictions`, `layer_pca`.
- `fct.tuning`: rates and latent factors (a task with factors, e.g. `linear_teacher`). `tuning_curves`, `select_neurons`.
- `fct.exp_view`: a single net on a task with factors. `tuning_view`, `layer_rates`, `neuron_information`.
- `fct.spatial`: needs a spatial preset (`spatial_tiny`). `build_spaces`, `latent_kernel`, `spatial_kernel_pairs`; for a net's data kernels use `fct.spatial_view.data_kernels(net, x, spaces)`.
- `fct.spatial_exp`: spatial experiments. `select_setting`, `load_exp_run`, `settings_table`.
- `fct.phase_view`: ensembles only (`net.B > 1`) and lattice or test sets. `layer_priors`, `member_view`, `ensemble_views`, `average_views`.
- `fct.draw`: plotting. `plot_loss_curves`, `fig_example_outputs`.

The R² of the trained net on its own training data (in-sample, not a held-out test).

In [ ]:
train_outputs = example_outputs(net, data, torch.arange(len(data.x)))
print("in-sample R^2:", r_squared(train_outputs.y_true, train_outputs.y_pred))

In [ ]:
fig, ax = plt.subplots()
ax.scatter(train_outputs.y_true.flatten(), train_outputs.y_pred.flatten(), s=4)
ax.set(xlabel="target", ylabel="network output")
plt.show()